# MYTourism Value Intelligence — Analytical Walkthrough

> **North-Star Principle**:
> *Do not only maximize tourists. Maximize sustainable economic value per visitor-day.*

This Jupyter notebook provides an interactive, reproducible walkthrough of the core analytical engines:
1. **Tourism Satellite Account (TSA)** Value-Added Intensity (VAI) Accounting
2. **State Economic Productivity** & Tourism Value-Added Yield (TVAY)
3. **Two-Way Fixed Effects Panel Econometrics**
4. **PPML Structural Gravity Modeling & 58 Pareto Corridors**
5. **Decoupled Scenario Simulation & Capacity Headroom**
6. **MILP Portfolio Budget Optimization**

In [ ]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

ROOT_DIR = Path("..").resolve()
DUCKDB_PATH = ROOT_DIR / "data/processed/tourism_data.duckdb"

con = duckdb.connect(str(DUCKDB_PATH), read_only=True)
print(f"Connected to DuckDB: {DUCKDB_PATH.name}")
tables = con.execute("SELECT count(*) FROM information_schema.tables WHERE table_schema='main'").fetchone()[0]
print(f"Total Analytical Tables Materialized: {tables}")

## 1. TSA Product Value-Added Intensity (VAI)
Gross Value Added divided by Domestic Supply across all 8 core tourism products (2015-2025).

In [ ]:
df_tsa = con.execute("""
    SELECT 
        product,
        post_recovery_median_vai,
        vai_2025,
        strategic_quadrant,
        itc_2025_rm_million,
        estimated_tourism_gva_2025_rm_million
    FROM product_value_summary
    ORDER BY post_recovery_median_vai DESC
""").df()
df_tsa


## 2. State Productivity & 4-Quadrant Typology
Length of stay vs. Tourism Value-Added Yield (TVAY) per visitor-day.

In [ ]:
df_state = con.execute("""
    SELECT 
        s.state,
        s.tourists_thousands,
        s.alos_days,
        s.spend_per_night_rm,
        sdg.tvay_rm_per_day,
        sdg.yield_typology
    FROM state_year s
    JOIN sdg_sustainable_metrics sdg ON s.state = sdg.state AND s.year = sdg.year
    WHERE s.year = 2025
    ORDER BY sdg.tvay_rm_per_day DESC
""").df()
df_state.head(10)

## 3. The 58 Non-Dominated Pareto Corridors
Corridors on Pareto Front 1 optimized across Demand Gap, TVAY, Capacity Headroom, HHI, and Accessibility.

In [ ]:
df_pareto = con.execute("""
    SELECT 
        corridor_id,
        origin,
        destination,
        actual_tourist_flow_thousands,
        gravity_flow_gap_thousands,
        dest_tvay_rm_per_day,
        dest_capacity_headroom_pct,
        composite_opportunity_score
    FROM corridor_opportunity_gap
    WHERE is_pareto_optimal = true
    ORDER BY composite_opportunity_score DESC
""").df()
print(f"Total Non-Dominated Front 1 Corridors: {len(df_pareto)}")
df_pareto.head(10)

## 4. Policy Scenario Simulation
Evaluating stay extension under physical hotel room constraints.

In [ ]:
import sys
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.scenarios.simulator import ScenarioSimulator
sim = ScenarioSimulator()

# Simulate +0.4 day stay extension on Selangor -> Melaka
res = sim.simulate_corridor('Selangor', 'Melaka', delta_alos=0.4, affected_share=0.15)
print("Scenario Output:")
print(f"  Additional Nights: {res['simulated_impact']['additional_tourist_nights_thousands']:,.1f} thousand")
print(f"  Additional Lodging Spend: RM {res['simulated_impact']['additional_accommodation_spend_rm_million']:.2f} Million")
print(f"  Potential GVA Proxy: RM {res['simulated_impact']['potential_additional_value_added_rm_million']:.2f} Million")
print(f"  Daily Rooms Demanded: {res['capacity_impact']['daily_room_demand_addition']:,.0f} rooms/day")
print(f"  Capacity Saturation Status: {res['capacity_impact']['capacity_status']}")
print(f"  Disclaimer: {res['disclaimer']}")

## 5. MILP Commercial Portfolio Optimization
Allocating a promotional budget of RM 5.0M across corridors to maximize expected GVA without breaching the 80% AOR ceiling.

In [ ]:
from src.scenarios.portfolio_optimizer import PortfolioOptimizer
opt = PortfolioOptimizer()
portfolio = opt.optimize_portfolio(budget_rm_million=5.0, planning_threshold=80.0)

print("MILP Portfolio Optimization Summary:")
print(f"  Status: {portfolio['status']}")
print(f"  Budget Utilized: RM {portfolio['summary']['total_cost_rm_million']:.2f}M / RM 5.00M")
print(f"  Expected Incremental GVA: RM {portfolio['summary']['total_expected_gva_rm_million']:.2f}M")
print(f"  Value-to-Cost Benchmark Multiple: {portfolio['summary']['portfolio_roi_multiplier']}x")
print(f"  Optimal Corridors Selected: {portfolio['summary']['corridors_selected_count']}")